# Module 08: Object Serialization with Pickle
## Notebook 03: Pickle Security Risks, Safe Unpickling & HMAC Authentication

The official Python documentation states:
> *"Warning: The `pickle` module is not secure. Only unpickle data you trust."*

Because Python's pickle engine is a stack-based virtual machine capable of constructing arbitrary callable objects, loading untrusted pickle files can result in **Remote Code Execution (RCE)**.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand the theoretical mechanism behind **Arbitrary Code Execution via Pickle**.
2. Construct a proof-of-concept malicious payload using the `__reduce__` exploit vector.
3. Build a secure **`RestrictedUnpickler`** that enforces strict whitelisting of safe classes and blocks dangerous builtins (`os`, `sys`, `eval`, `subprocess`).
4. Implement **HMAC Cryptographic Signing (SHA-256)** to verify data integrity and authenticity before deserialization.
5. Know when to use **JSON**, **Parquet**, or **Safetensors** instead of Pickle in production pipelines.

In [ ]:
import os
import io
import pickle
import hmac
import hashlib

print("Security sandbox loaded.")

### 1. Anatomy of the Pickle Exploit
When `pickle.load()` encounters an object defined with `__reduce__()`, it resolves the callable and executes it with the provided arguments.
If an attacker crafts a payload where the callable is `os.system` or `subprocess.Popen`, the unpickler will execute the attacker's shell commands **with the full privileges of the running Python process**!

In [ ]:
# Exploit Demonstration: Constructing a harmless proof-of-concept payload
class HarmlessExploitProof:
    def __reduce__(self):
        # We invoke print() instead of malicious commands to demonstrate execution
        return (print, ("\n[SECURITY ALERT] Code execution triggered during unpickling!\n",))

import __main__
__main__.HarmlessExploitProof = HarmlessExploitProof

# Attacker creates and serializes payload
malicious_payload = pickle.dumps(HarmlessExploitProof())
print(f"Generated Payload: {len(malicious_payload)} bytes")

# Victim innocently calls pickle.loads() on untrusted payload
print("Simulating victim calling pickle.loads(payload)...")
_ = pickle.loads(malicious_payload)

### 2. Defensive Strategy 1: Whitelisted Restricted Unpickler
To safely unpickle data, we subclass `pickle.Unpickler` and override **`find_class(module, name)`**.
- Standard `Unpickler.find_class()` imports any requested module (`os`, `subprocess`).
- `RestrictedUnpickler.find_class()` checks requested classes against an explicit **allowlist** (e.g. `numpy`, `builtins.int`, `builtins.float`).
- Any attempt to import unauthorized modules immediately raises a `pickle.UnpicklingError`!

In [ ]:
class RestrictedUnpickler(pickle.Unpickler):
    # Safe unpickler enforcing an explicit allowlist of authorized classes.
    SAFE_ALLOWLIST = {
        ("numpy", "ndarray"),
        ("numpy", "dtype"),
        ("numpy._core.multiarray", "_reconstruct"),
        ("numpy._core.multiarray", "scalar"),
        ("builtins", "int"),
        ("builtins", "float"),
        ("builtins", "str"),
        ("builtins", "list"),
        ("builtins", "dict"),
        ("builtins", "set"),
        ("builtins", "tuple"),
        ("builtins", "bool"),
    }

    def find_class(self, module, name):
        # Only allow authorized classes from allowlist
        if (module, name) in self.SAFE_ALLOWLIST:
            return super().find_class(module, name)
        # Deny all others
        raise pickle.UnpicklingError(f"Security Violation: Unauthorized global '{module}.{name}' blocked!")

def safe_loads(byte_data):
    return RestrictedUnpickler(io.BytesIO(byte_data)).load()

# Test 1: Safe legitimate object
safe_obj = {"scores": [98.5, 91.2], "model_name": "ResNet"}
safe_bytes = pickle.dumps(safe_obj)
restored_safe = safe_loads(safe_bytes)
print(f"Test 1 (Safe Object): Successfully unpickled -> {restored_safe}")

# Test 2: Malicious exploit payload
print("\nTest 2 (Malicious Payload): Attempting safe unpickling...")
try:
    safe_loads(malicious_payload)
except pickle.UnpicklingError as e:
    print(f"DEFENSE SUCCESSFUL: Blocked exploit attempt!\n -> {e}")

### 3. Defensive Strategy 2: Cryptographic HMAC Signature Verification
In production model pipelines (e.g. saving models to S3 or transferring models across worker nodes), we verify artifact authenticity using **HMAC (Hash-based Message Authentication Code)**:
1. The model producer serializes the object and computes an HMAC signature using a shared secret key:
   $$\text{Signature} = \text{HMAC-SHA256}(K, \text{Payload})$$
2. The consumer verifies the signature *before* calling `pickle.loads()`.
3. If an attacker tampers with a single bit in the pickle file, `hmac.compare_digest()` fails and the file is discarded before unpickling!

In [ ]:
SECRET_KEY = b"production-secret-key-38914"

class SecurityError(Exception):
    pass

def sign_and_serialize(obj, secret_key):
    # Serialize object
    payload = pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
    # Compute HMAC-SHA256 signature
    signature = hmac.new(secret_key, payload, hashlib.sha256).digest()
    # Prepend 32-byte signature to payload
    return signature + payload

def verify_and_deserialize(signed_data, secret_key):
    SIG_SIZE = 32 # SHA256 produces 32 bytes
    if len(signed_data) < SIG_SIZE:
        raise ValueError("Invalid signed payload: Too short!")

    # Extract signature and payload
    expected_sig = signed_data[:SIG_SIZE]
    payload = signed_data[SIG_SIZE:]

    # Compute actual signature
    actual_sig = hmac.new(secret_key, payload, hashlib.sha256).digest()

    # Constant-time comparison to prevent timing attacks
    if not hmac.compare_digest(expected_sig, actual_sig):
        raise SecurityError("HMAC Signature Mismatch! Payload has been tampered with or corrupted.")

    return pickle.loads(payload)

# 1. Producer signs and saves model
model_state = {"weights": [0.2, 0.4, 0.6], "status": "verified"}
signed_package = sign_and_serialize(model_state, SECRET_KEY)
print(f"Signed Package Size: {len(signed_package)} bytes (32-byte HMAC prefix)")

# 2. Legitimate consumer verifies and loads
verified_obj = verify_and_deserialize(signed_package, SECRET_KEY)
print(f"Consumer Verification Successful: {verified_obj}")

# 3. Attacker intercepts and tampers with payload
tampered_package = bytearray(signed_package)
tampered_package[45] ^= 0xFF # Flip bits in the serialized payload

print("\nSimulating tampered payload receipt:")
try:
    verify_and_deserialize(bytes(tampered_package), SECRET_KEY)
except SecurityError as e:
    print(f"DEFENSE SUCCESSFUL: {e}")